In [1]:
import pandas as pd
import numpy as np
df=pd.read_csv("Telco_customer_churn.csv")
print("shape fo the data ",df.shape)
print("columns of the data ",df.columns)
print("sata types ",df.dtypes)
print("null values ",df.isnull().sum())
print(df["Churn Label"].value_counts())

shape fo the data  (7043, 33)
columns of the data  Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='object')
sata types  CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents          

In [2]:
df["Total Charges"].dtypes

dtype('O')

In [3]:
df["Total Charges"]=pd.to_numeric(df["Total Charges"],errors="coerce")

In [4]:
df["Total Charges"].isnull().sum()

np.int64(11)

In [5]:
df["Total Charges"]=df["Total Charges"].fillna(df["Total Charges"].median())

In [6]:
df["Total Charges"].median()


np.float64(1397.475)

In [7]:
drop_columns=["CustomerID","Country","State","City","Lat Long","Latitude","Longitude","Zip Code","Churn Score","Churn Label","Churn Reason","Count"]


In [8]:
model_df=df.drop(columns=drop_columns)

In [9]:
model_df.head(2)

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701


In [10]:
model_df["Avg_monthSpend"]=model_df["Total Charges"]/model_df["Tenure Months"]+1

In [11]:

model_df["Num Service"] =( model_df["Phone Service"].apply(lambda x: 1 if x == "Yes" else 0)+ 
model_df["Multiple Lines"].apply(lambda x: 1 if x=="Yes" else 0)+
model_df["Online Security"].apply(lambda x : 1 if x=="Yes" else 0)+
model_df["Online Backup"].apply(lambda x: 1 if x=="Yes" else 0)+
model_df["Device Protection"].apply(lambda x: 1 if x=="Yes" else 0)+
model_df["Tech Support"].apply(lambda x :1 if x=="Yes" else 0)+
model_df["Streaming TV"].apply(lambda x:1 if x =="Yes" else 0)+
model_df["Streaming Movies"].apply(lambda x: 1 if x=="Yes" else 0))

In [12]:
print(model_df["Num Service"].value_counts().sort_index())

Num Service
0      80
1    1701
2    1188
3     965
4     922
5     908
6     676
7     395
8     208
Name: count, dtype: int64


In [13]:
model_df["Is Month to Month"]=model_df["Contract"].apply(lambda x: 1 if x =="Month-to-month"else 0 )
model_df["Fiber"]=model_df["Internet Service"].apply(lambda x: 1 if x =="Fiber optic" else 0)

In [14]:
print(model_df["Is Month to Month"].value_counts())
print(model_df["Fiber"].value_counts())

Is Month to Month
1    3875
0    3168
Name: count, dtype: int64
Fiber
0    3947
1    3096
Name: count, dtype: int64


In [15]:
model_df.select_dtypes(include="string").columns



Index([], dtype='object')

In [16]:
from sklearn.preprocessing import LabelEncoder




In [17]:
encoder=LabelEncoder()

In [18]:
for col in model_df.select_dtypes(include="string").columns:
    model_df[col]=encoder.fit_transform(model_df[col])

In [19]:
x=model_df.drop(["Churn Value"],axis =1)
y=model_df["Churn Value"]

In [20]:
from sklearn.model_selection import train_test_split 

In [21]:
x_train,x_test,y_train, y_test=train_test_split(x,y,test_size=0.2 ,stratify=y,random_state=42)

In [22]:
print(x_train.shape)
print(x_test.shape)

(5634, 24)
(1409, 24)


In [23]:
print(x_train.isnull().sum()[x_train.isnull().sum() > 0])

Series([], dtype: int64)


In [24]:
import numpy as np
x_train=x_train.replace([np.inf,-np.inf], np.nan)
x_test=x_test.replace([np.inf,-np.inf],np.nan)
for col in x_train.select_dtypes(include='object').columns:
    x_train[col] = encoder.fit_transform(x_train[col].astype(str))
    x_test[col]  = encoder.transform(x_test[col].astype(str))

x_train.fillna(x_train.median())
x_test.fillna(x_test.median())

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,CLTV,Avg_monthSpend,Num Service,Is Month to Month,Fiber
2196,1,0,1,0,72,1,2,1,2,2,...,2,1,1,114.05,8468.20,4842,118.613889,8,0,1
3549,0,1,0,0,8,1,2,1,0,0,...,0,1,1,100.15,908.55,5157,114.568750,5,1,1
3515,0,0,1,0,41,1,2,0,2,2,...,1,1,1,78.35,3211.20,2894,79.321951,6,0,0
5162,1,0,1,0,18,1,0,1,0,0,...,0,0,2,78.20,1468.75,2831,82.597222,3,1,1
4642,0,0,1,0,72,1,2,0,2,2,...,2,1,1,82.65,5919.35,4324,83.213194,7,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5660,0,0,1,1,49,1,0,0,2,2,...,1,1,3,87.20,4345.00,4567,89.673469,7,0,0
5150,1,0,1,1,28,1,0,2,1,1,...,2,1,1,20.30,487.95,2948,18.426786,1,0,0
4708,1,0,0,0,5,1,0,2,1,1,...,0,0,0,20.65,93.55,5597,19.710000,1,1,0
5381,0,0,0,0,56,1,0,2,1,1,...,2,0,0,19.70,1051.90,4831,19.783929,1,0,0


In [25]:
from sklearn.ensemble import RandomForestClassifier

In [26]:
rf=RandomForestClassifier()

In [27]:
rf.fit(x_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [28]:
y_pred=rf.predict(x_test)

In [29]:
y_pred

array([0, 1, 0, ..., 0, 0, 0], shape=(1409,))

In [30]:
from sklearn.metrics import classification_report

In [31]:
cr=classification_report(y_test,y_pred)

In [39]:
print(cr)

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [33]:
y_prob=rf.predict_proba(x_test)[:,1]

In [38]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [43]:
from sklearn.metrics import roc_auc_score

In [45]:
print("Roc_auc",roc_auc_score(y_test,y_prob))

Roc_auc 0.8367614766591749


In [37]:
import pickle

# Save the model
with open("churn_model.pkl", "wb") as f:
    pickle.dump(rf, f)

# Save the scaler too (needed for preprocessing)
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and scaler saved!")
```

This creates two files in the same folder as your notebook:
- `churn_model.pkl` — your trained Random Forest
- `scaler.pkl` — your StandardScaler

Run that and tell me it printed "Model and scaler saved!" 👇

---

### While You Do That — Here's The Plan

Once saved, we'll create one file called `app.py` in the same folder. The structure will be:
```
Your Project Folder/
│
├── churn_model.pkl      ← model you're saving now
├── scaler.pkl           ← scaler you're saving now
├── app.py               ← Streamlit app we'll build next
└── your_notebook.ipynb  ← your existing notebook

array([0, 1, 0, ..., 0, 0, 0], shape=(1409,))

In [46]:
import pickle

In [49]:
with open("churn-prediction","wb") as f:
    pickle.dump(rf ,f)
print("model  saved")



model  saved
